In [29]:
# Fuzzing Assignment Notebook
# Author: Shivani Poosarla
# Course: Automated Techniques in Security, Privacy and Reliability
# Description: This notebook follows the Fuzzingbook tutorials to complete the fuzzing assignment.

In [30]:
!apt-get install -y graphviz libgraphviz-dev pkg-config
!pip install pygraphviz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pkg-config is already the newest version (0.29.2-1ubuntu3).
graphviz is already the newest version (2.42.2-6ubuntu0.1).
libgraphviz-dev is already the newest version (2.42.2-6ubuntu0.1).
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [31]:
# Install fuzzingbook
!pip install fuzzingbook

# Imports from fuzzingbook
from fuzzingbook.Fuzzer import *
from fuzzingbook.Coverage import *
import matplotlib.pyplot as plt
from fuzzingbook.MutationFuzzer import MutationFuzzer

Part 1: Basic Fuzzing

In [32]:
from fuzzingbook.Fuzzer import RandomFuzzer

# Define a custom random fuzzer by extending RandomFuzzer
# This fuzzer generates random strings based on default parameters
class MyRandomFuzzer(RandomFuzzer):
    def fuzz(self):
        # Use the parent class's fuzzing behavior (can customize if needed)
        return super().fuzz()

# Example function to fuzz
# If the input string matches "crash", it throws an exception.
# The fuzzer's goal is to eventually generate that input (or something else that breaks the function).
def my_test_function(input_str):
    if input_str == "crash":
        raise Exception("Crash!")  # Simulate a bug
    return input_str  # Otherwise, return the input as-is

# Instantiate the fuzzer and run it
fuzzer = MyRandomFuzzer()
for i in range(10):  # Run 10 fuzzing iterations
    inp = fuzzer.fuzz()
    try:
        # Try running the function with the fuzzed input
        print(my_test_function(inp))
    except Exception as e:
        # Catch and print any exceptions that occur
        print(f"Caught exception: {e}")


(/3&(8"-"9/;$.<<:&,/%<"3?0/5(*,63;5)>:0)19,52&?<?(0#27(8&!9(3
<!= 4<2:56,4$*/:>>>2%<1-9-8>97"$>0.!-$/)2""("92 >,").&=37?38"%9'5:&?$1./,3#.$
>7&6-!%*$)=+$15%>)=5#)64;0+&7-8,><4 3"+36231"*$=)>*5593&
0 .(!>4211;/3*=>2$'25)%)6&1>:;=
(29&+(-.,,':>7(9(;:3?! 2")&=#1<)<&4,(#'#=3+5%!$"
8"(-"#;=,?>:$,//13=*>9: -#<673!<
-#60$-17:=#'49;48(!.)%?&8*#  < - 7?!05%5-1.#('",7
%>==.(4#3%1=:%(=95 =#=
6'#? (?2$< +,<45&(&:=/52 0!1!8#>'??
()<;< 4-<"+.''=<83::-=)0)+#("873%#=>".94;:3'>4;)( *2&4(<&(<,8#!#)6*;+6-%*)(".0!5783'4$6263'907)4'9


Part 1 Submission: Sanitizer Research

In [33]:
#Research on Two Clang Sanitizers (other than AddressSanitizer)

"""
1. MemorySanitizer (MSan):
   - MemorySanitizer is designed to detect the use of uninitialized memory in C/C++ programs.
   - This includes cases where a variable is read before any value has been written to it.
   - It's especially helpful for tracking down subtle bugs that might lead to unpredictable behavior.
   - To enable it during compilation with clang, use the flag: -fsanitize=memory

2. UndefinedBehaviorSanitizer (UBSan):
   - UBSan helps catch instances of undefined behavior in C/C++ code.
   - This includes issues like signed integer overflow, division by zero, misaligned memory access, or using a null pointer.
   - It provides runtime checks to make these problems easier to identify during testing.
   - To enable UBSan, compile with the flag: -fsanitize=undefined
"""

"\n1. MemorySanitizer (MSan):\n   - MemorySanitizer is designed to detect the use of uninitialized memory in C/C++ programs.\n   - This includes cases where a variable is read before any value has been written to it.\n   - It's especially helpful for tracking down subtle bugs that might lead to unpredictable behavior.\n   - To enable it during compilation with clang, use the flag: -fsanitize=memory\n\n2. UndefinedBehaviorSanitizer (UBSan):\n   - UBSan helps catch instances of undefined behavior in C/C++ code.\n   - This includes issues like signed integer overflow, division by zero, misaligned memory access, or using a null pointer.\n   - It provides runtime checks to make these problems easier to identify during testing.\n   - To enable UBSan, compile with the flag: -fsanitize=undefined\n"

Part 2: Coverage

In [34]:
from fuzzingbook.Coverage import Coverage

# Sample target function with multiple branches
# This function returns different values depending on the input string.
# We'll use it to test how much of the code gets covered during execution.
def bc(x):
    if x == "foo":
        return 1
    elif x == "bar":
        return 2
    elif x.startswith("ba"):
        return 3
    return 0

# Create a Coverage object to track executed lines
cov = Coverage()

# Execute the function with various inputs while tracking coverage
with cov:
    bc("foo")     # Should hit the first branch
    bc("bar")     # Should hit the second branch
    bc("baz")     # Should match the third condition (starts with 'ba')
    bc("hello")   # Should fall through to the default case

# Print the full execution trace (function name + line numbers)
trace = cov.trace()
print("Trace:", trace)

# Get the coverage data as a set of (function_name, line_number) tuples
coverage = cov.coverage()
print("Raw coverage set:", coverage)

# Filter out just the lines from the function 'bc'
lines_covered = {lineno for (func, lineno) in coverage if func == 'bc'}
print("Lines covered in bc():", lines_covered)


Trace: [('bc', 7), ('bc', 8), ('bc', 7), ('bc', 9), ('bc', 10), ('bc', 7), ('bc', 9), ('bc', 11), ('bc', 12), ('bc', 7), ('bc', 9), ('bc', 11), ('bc', 13), ('_internal_set_trace', 48), ('_internal_set_trace', 49), ('_internal_set_trace', 50), ('_internal_set_trace', 51), ('splitext', 118), ('splitext', 119), ('splitext', 123), ('splitext', 124), ('splitext', 125), ('_splitext', 128), ('_splitext', 129), ('_splitext', 133), ('_splitext', 134), ('_splitext', 136), ('_splitext', 137), ('_splitext', 138), ('_splitext', 139), ('_internal_set_trace', 52), ('_internal_set_trace', 57), ('_internal_set_trace', 58), ('_internal_set_trace', 57), ('_internal_set_trace', 65), ('_internal_set_trace', 68), ('_get_stack_str', 34), ('_get_stack_str', 38), ('_get_stack_str', 39), ('_get_stack_str', 40), ('_get_stack_str', 41), ('print_stack', 205), ('print_stack', 207), ('extract_stack', 226), ('extract_stack', 228), ('extract', 386), ('extract', 390), ('extract', 391), ('extract', 392), ('extract', 390

Part 3: Mutation Strategies

Exercise 2 – Custom Mutation Strategy

In [35]:
import random
import string
from fuzzingbook.MutationFuzzer import MutationFuzzer

# CustomMutationFuzzer: Subclass of MutationFuzzer
# This fuzzer overrides the mutate() method to implement a simple character replacement.
# The strategy randomly selects a character in the input and replaces it with a random printable character.
class CustomMutationFuzzer(MutationFuzzer):
    def mutate(self, inp):
        # If input is not empty, replace a random character
        if len(inp) > 0:
            i = random.randrange(len(inp))  # Pick a random position
            return inp[:i] + random.choice(string.printable) + inp[i+1:]
        else:
            # If input is empty, return a random printable character
            return random.choice(string.printable)

# Test the custom fuzzer with a few seed inputs
inputs = ["foo", "bar", "baz"]
fuzzer = CustomMutationFuzzer(inputs)

# Generate and print a few fuzzed inputs
for _ in range(10):
    print(fuzzer.fuzz())


foo
bar
baz
fo=
FvD
b %
fo/
WI
V>t
V'r


Exercise 3 – AFL Mutation Strategies

In [36]:
import random

# AFL Mutation Strategy 1: Bitflip
# This function flips a single random bit in one byte of the input string.
# Bitflipping is useful to trigger edge cases in input parsing (e.g., off-by-one errors).
def afl_bitflip(data):
    if len(data) == 0:
        return data  # Nothing to flip if input is empty

    # Pick a random byte index in the string
    i = random.randrange(len(data))

    # Get the ASCII value of the character at that index
    b = ord(data[i])

    # Flip the least significant bit (AFL flips individual bits in multiple strategies)
    b ^= 0x01

    # Reconstruct the mutated string with the bit-flipped byte
    return data[:i] + chr(b) + data[i+1:]

# AFL Mutation Strategy 2: Known Integer Insertion
# This strategy appends a known "interesting" integer to the input.
# These values are known to often trigger integer-related bugs in software.
def afl_known_integer(data):
    known_ints = ["0", "1", "255", "32767", "2147483647"]

    # Randomly choose one of the known integers and append it
    return data + random.choice(known_ints)

# Example usage of the two AFL-inspired strategies on seed inputs
for s in ["hello", "world"]:
    print("Original:", s)
    print("Bitflip:", afl_bitflip(s))
    print("Known Int:", afl_known_integer(s))


Original: hello
Bitflip: iello
Known Int: hello0
Original: world
Bitflip: wosld
Known Int: world2147483647


Exercise 4 Evaluation of Mutation Efficiency

In [37]:
from fuzzingbook.Coverage import Coverage
from fuzzingbook.MutationFuzzer import MutationFuzzer

# Target function to fuzz
# This function has multiple branches based on the input string.
# The goal is to see how many of these branches get exercised during fuzzing.
def bc(x):
    if x == "foo":
        return 1
    elif x == "bar":
        return 2
    elif x.startswith("ba"):
        return 3
    return 0

# Set up a Coverage object to track which lines are executed
cov = Coverage()

# Provide a set of seed inputs for the fuzzer to mutate
seed_inputs = ["foo", "bar", "baz"]
fuzzer = MutationFuzzer(seed_inputs)

# Run the fuzzer and track coverage
# Each input is generated by mutating the seed set
# and passed to the target function under coverage monitoring
with cov:
    for _ in range(100):  # Run 100 fuzzing iterations
        fuzz_input = fuzzer.fuzz()
        bc(fuzz_input)  # Call the function with the fuzzed input
        print(f"Fuzzed input: {fuzz_input}")

# After fuzzing, print out which functions and lines were covered
# This shows how effective the mutation strategy was at exploring code paths
coverage_by_function = cov.coverage()
print("Coverage by function:", coverage_by_function)


Fuzzed input: foo
Fuzzed input: bar
Fuzzed input: baz
Fuzzed input: cHP
Fuzzed input: BJar
Fuzzed input: o
Fuzzed input: 9)m
Fuzzed input: bcr]0
Fuzzed input: r6go
Fuzzed input: Z
Fuzzed input: 
Fuzzed input: "f
Fuzzed input: 9,GYVb
Fuzzed input: b;
Fuzzed input: 
Fuzzed input: qw
Fuzzed input: _fv
Fuzzed input: o.
Fuzzed input: TnSf
Fuzzed input: ro
Fuzzed input: 
Fuzzed input: bj
Fuzzed input: brG
Fuzzed input: oy
Fuzzed input: 
Fuzzed input: azlH
Fuzzed input: bva[^
Fuzzed input: n*Sn
Fuzzed input: b}Gp
Fuzzed input: d]S\>Fooh
Fuzzed input: "T
Fuzzed input: \
Fuzzed input: baQF
Fuzzed input: eAr
Fuzzed input: 3*no
Fuzzed input: ]c%n8
Fuzzed input: o
Fuzzed input: bar
Fuzzed input: 7o
Fuzzed input: 
Fuzzed input: 
Fuzzed input: 
Fuzzed input: o
Fuzzed input: o~
Fuzzed input: *a
Fuzzed input: z
Fuzzed input: no
Fuzzed input: Yfw
Fuzzed input: p[{
Fuzzed input: c0bn
Fuzzed input: 3Uf
Fuzzed input: N`0a
Fuzzed input: o|got
Fuzzed input: q a
Fuzzed input: bhz
Fuzzed input: 
Fuzzed inp